# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
import glob

# Works for 'AviationData.csv' as well as a copy such as 'AviationData (1).csv'
data_path = glob.glob('AviationData*.csv')[0]

# The file contains some non-UTF-8 characters, so latin-1 encoding is used
df = pd.read_csv(data_path, encoding='latin-1', low_memory=False)

print(df.shape)
df.info()
print()
print(df.isna().sum().sort_values(ascending=False))
df.describe()

(88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# 1. Dates: convert to datetime and keep 1983 onwards (40-year maximum lifetime of a make/model)
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1983]
print("1983 onwards:", len(df))

# 2. Only keep full 'Accident' investigations (incidents are minor events with little damage/injury information)
df = df[df['Investigation.Type'] == 'Accident']
print("Accidents only:", len(df))

# 3. Only professional builds: drop amateur-built aircraft and the few rows where this is unknown
df = df.dropna(subset=['Amateur.Built'])
df = df[df['Amateur.Built'] == 'No'].copy()
print("Professionally built:", len(df))

# 4. Standardise Make and Model text (needed for the imputation below) and drop rows without them
df['Make'] = df['Make'].str.strip().str.upper()
df['Model'] = df['Model'].str.strip().str.upper()
df = df.dropna(subset=['Make', 'Model'])

# 5. Aircraft.Category is missing for most rows, mostly older records.
#    Imputation: if the same Make + Model has a known category elsewhere in the data, use its most common category.
df['Aircraft.Category'] = df['Aircraft.Category'].replace({'Unknown': np.nan, 'UNK': np.nan})
known = df.dropna(subset=['Aircraft.Category'])
category_lookup = known.groupby(['Make', 'Model'])['Aircraft.Category'].agg(lambda s: s.mode().iloc[0])

keys = pd.MultiIndex.from_frame(df[['Make', 'Model']])
looked_up = pd.Series(category_lookup.reindex(keys).values, index=df.index)
missing = df['Aircraft.Category'].isna()
df.loc[missing, 'Aircraft.Category'] = looked_up[missing]

# Remaining gaps: turbine engines (fan, jet, prop) on fixed-wing aircraft are airplanes
# (helicopters use turbo shaft engines), so they can be safely labelled 'Airplane'
turbine_airplane = df['Aircraft.Category'].isna() & df['Engine.Type'].isin(['Turbo Fan', 'Turbo Jet', 'Turbo Prop'])
df.loc[turbine_airplane, 'Aircraft.Category'] = 'Airplane'

# 6. The client only cares about airplanes (no helicopters, gliders, balloons, ...); rows still unlabelled are dropped
print(df['Aircraft.Category'].value_counts(dropna=False))
df = df[df['Aircraft.Category'] == 'Airplane'].copy()
print("Airplanes:", len(df))

1983 onwards: 85289
Accidents only: 81534
Professionally built: 73286
Aircraft.Category
Airplane             59134
Helicopter            6464
NaN                   6284
Glider                 777
Balloon                298
Weight-Shift           139
Powered Parachute       83
Gyrocraft               31
Ultralight              18
WSFT                     9
Blimp                    5
Powered-Lift             4
Rocket                   1
Name: count, dtype: int64
Airplanes: 59134


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Assumption 1: when Total.Fatal.Injuries is missing but Injury.Severity says e.g. 'Fatal(3)', use that number
fatal_from_severity = df['Injury.Severity'].str.extract(r'Fatal\((\d+)\)')[0].astype(float)
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(fatal_from_severity)

# Assumption 2: the remaining missing injury counts mean nobody was in that category, so they are set to 0
df[injury_cols] = df[injury_cols].fillna(0)

# Estimated number of people on board = fatal + serious + minor + uninjured
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# Rows where no one is recorded on board cannot give an injury fraction, so they are dropped
df = df[df['Total.Passengers'] > 0].copy()

# Fraction of people on board who were fatally or seriously injured
df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

print(df.shape)
df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe()

(58816, 33)


,Total.Passengers,Fatal.Serious.Fraction
count,58816.000000,58816.000000
mean,4.983423,0.277336
std,23.674774,0.431739
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,2.000000,0.750000
max,699.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
# Tidy the labels and treat 'Unknown' as missing
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title().replace({'Unknown': np.nan})
print(df['Aircraft.damage'].value_counts(dropna=False))

# Damage is the second key outcome, so rows without it cannot be used and are dropped
df = df.dropna(subset=['Aircraft.damage']).copy()

# Derived column: 1 if the aircraft was destroyed, 0 otherwise (Substantial / Minor)
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
df['Destroyed'].mean()

Aircraft.damage
Substantial    45313
Destroyed      12079
NaN              985
Minor            439
Name: count, dtype: int64


0.20886721654475973

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Make cleaning tasks identified**
1. Upper-case and strip whitespace so `Cessna` and `CESSNA` are the same.
2. Merge name variants of the same manufacturer, e.g. `CESSNA AIRCRAFT CO`, `PIPER AIRCRAFT INC`, `GRUMMAN AMERICAN`, `DEHAVILLAND` and `DE HAVILLAND`, `AIRBUS INDUSTRIE` and `AIRBUS`.
3. Merge manufacturers that are the same company line: `CANADAIR` with `BOMBARDIER`, `GATES LEARJET` with `LEARJET`, `HAWKER BEECHCRAFT` with `BEECH`.
4. Keep the Boeing-Stearman biplane separate from `BOEING` (jets): both the make `BOEING STEARMAN` and `BOEING` rows with Stearman models (Model 75, PT-17, N2S) become `STEARMAN`, so small aircraft do not distort the Boeing results.
5. Keep only makes with at least 50 accidents.

In [6]:
import re

def clean_make(name):
    # remove punctuation and extra spaces
    name = re.sub(r'[.,()]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    # (prefix, standard name): the first matching prefix wins, so more specific prefixes come first
    rules = [
        ('BOEING STEARMAN', 'STEARMAN'), ('BOEING-STEARMAN', 'STEARMAN'), ('BOEING (STEARMAN)', 'STEARMAN'),
        ('BOEING CANADA', 'DE HAVILLAND'), ('BOEING - CANADA', 'DE HAVILLAND'), ('BOEING OF CANADA', 'DE HAVILLAND'),
        ('THE BOEING', 'BOEING'), ('BOEING', 'BOEING'),
        ('REIMS-CESSNA', 'CESSNA'), ('REIMS CESSNA', 'CESSNA'), ('CESSNA', 'CESSNA'),
        ('NEW PIPER', 'PIPER'), ('PIPER', 'PIPER'),
        ('HAWKER BEECH', 'BEECH'), ('BEECH', 'BEECH'),
        ('MOONEY', 'MOONEY'), ('GRUMMAN', 'GRUMMAN'), ('GULFSTREAM', 'GULFSTREAM'),
        ('AMERICAN CHAMPION', 'CHAMPION'), ('DEHAVILLAND', 'DE HAVILLAND'),
        ('NORTH AMERICAN ROCKWELL', 'ROCKWELL'), ('ROCKWELL', 'ROCKWELL'),
        ('MCDONNELL DOUGLAS', 'MCDONNELL DOUGLAS'), ('CIRRUS', 'CIRRUS'),
        ('EMBRAER', 'EMBRAER'), ('AIRBUS', 'AIRBUS'),
        ('GATES LEARJET', 'LEARJET'), ('LEARJET', 'LEARJET'),
        ('FAIRCHILD SWEARINGEN', 'SWEARINGEN'), ('SINO-SWEARINGEN', 'SWEARINGEN'), ('SWEARINGEN', 'SWEARINGEN'),
        ('DIAMOND', 'DIAMOND'), ('RAYTHEON', 'RAYTHEON'),
        ('CANADAIR', 'BOMBARDIER'), ('BOMBARDIER', 'BOMBARDIER'),
        ('EADS SOCATA', 'SOCATA'), ('SOCATA', 'SOCATA'),
        ('AIR TRACTOR', 'AIR TRACTOR'), ('ERCOUPE', 'ERCOUPE'), ('SAAB', 'SAAB'),
        ('LOCKHEED', 'LOCKHEED'), ('DORNIER', 'DORNIER'), ('AEROSPATIALE', 'AEROSPATIALE'),
    ]
    if name == 'AVIAT' or name.startswith('AVIAT '):
        return 'AVIAT'
    for prefix, standard in rules:
        if name.startswith(prefix):
            return standard
    return name

print("Unique makes before:", df['Make'].nunique())
df['Make'] = df['Make'].apply(clean_make)
print("Unique makes after:", df['Make'].nunique())

# 'BOEING' also contains the small Boeing-Stearman biplane (Model 75, PT-17, N2S), which are not Boeing jets
stearman = (df['Make'] == 'BOEING') & df['Model'].str.match(r'^(A75|B75|D75|E75|A-75|N2S|PT|75$|75N|75-)')
print("Boeing-Stearman rows moved to STEARMAN:", stearman.sum())
df.loc[stearman, 'Make'] = 'STEARMAN'

# Keep makes with enough accidents (threshold 50) so that averages are meaningful
make_counts = df['Make'].value_counts()
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)].copy()
print("Makes kept:", df['Make'].nunique(), "| rows:", len(df))
df['Make'].value_counts()

Unique makes before: 1076
Unique makes after: 919
Boeing-Stearman rows moved to STEARMAN: 296
Makes kept: 47 | rows: 54990


Make
CESSNA               24498
PIPER                13447
BEECH                 4431
GRUMMAN               1359
MOONEY                1204
BELLANCA               909
AIR TRACTOR            863
CHAMPION               560
AERONCA                527
MAULE                  488
BOEING                 483
CIRRUS                 426
STINSON                390
DE HAVILLAND           377
LUSCOMBE               362
STEARMAN               336
ROCKWELL               330
AERO COMMANDER         319
TAYLORCRAFT            316
NORTH AMERICAN         294
AVIAT                  221
ERCOUPE                206
AYRES                  203
LEARJET                181
MCDONNELL DOUGLAS      164
DIAMOND                159
SWEARINGEN             128
SCHWEIZER              127
GULFSTREAM             126
LAKE                   122
DOUGLAS                121
MITSUBISHI             121
EMBRAER                114
BOMBARDIER             104
SOCATA                 103
WACO                    97
RAYTHEON               

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

**Model notes:** a label like `172` or `A36` can be shared across manufacturers, so averages by model alone could mix different aircraft. A combined `Plane.Type` (Make + Model) is used instead.

In [7]:
# NaNs in Model were already removed and the text standardised (strip + upper case) when filtering.
# Model labels are NOT unique to a make: e.g. the same label can be used by several manufacturers
shared = df.groupby('Model')['Make'].nunique()
print("Model labels used by more than one make:", (shared > 1).sum())
print(shared.sort_values(ascending=False).head())

# So a unique identifier for each plane type is built by combining Make and Model
df['Plane.Type'] = df['Make'] + ' ' + df['Model']
print("Unique plane types:", df['Plane.Type'].nunique())
df['Plane.Type'].value_counts().head(10)

Model labels used by more than one make: 121
Model
G-164C      3
690B        3
DC-10-30    3
G-164D      3
S2R         3
Name: Make, dtype: int64
Unique plane types: 2557


Plane.Type
CESSNA 152         2200
CESSNA 172         1611
CESSNA 172N        1080
PIPER PA-28-140     856
CESSNA 172M         751
CESSNA 150          739
CESSNA 172P         659
CESSNA 182          602
CESSNA 180          592
PIPER PA-18-150     553
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
# Engine.Type: 'Unknown', 'UNK', 'NONE' and 'LR' carry no information -> NaN
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'NONE': np.nan, 'LR': np.nan})

# Weather.Condition: standardise case, 'UNK' / 'Unk' -> NaN (leaves VMC and IMC)
df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace({'UNK': np.nan, 'UNKNOWN': np.nan})

# Number.of.Engines: an airplane cannot have 0 engines, so 0 is an error -> NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace({0: np.nan})

# Purpose.of.flight: merge duplicate labels / codes and combine the public-aircraft variants
df['Purpose.of.flight'] = (df['Purpose.of.flight'].str.strip()
                           .replace({'Unknown': np.nan, 'PUBS': 'Public Aircraft',
                                     'Public Aircraft - Federal': 'Public Aircraft',
                                     'Public Aircraft - State': 'Public Aircraft',
                                     'Public Aircraft - Local': 'Public Aircraft',
                                     'Air Race show': 'Air Race/show', 'ASHO': 'Air Race/show'}))

# Broad.phase.of.flight: 'Unknown' -> NaN
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().replace({'Unknown': np.nan})

# NaNs are left in place here: they can simply be left out of any comparison for that factor
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False), '\n')

Engine.Type
Reciprocating    48387
NaN               3049
Turbo Prop        2424
Turbo Fan          866
Turbo Jet          253
Turbo Shaft         11
Name: count, dtype: int64 

Weather.Condition
VMC    48480
IMC     4390
NaN     2120
Name: count, dtype: int64 

Number.of.Engines
1.0    45885
2.0     6811
NaN     2033
4.0      137
3.0      124
Name: count, dtype: int64 

Purpose.of.flight
Personal               32812
Instructional           7720
NaN                     5235
Aerial Application      3170
Business                2677
Positioning              923
Other Work Use           484
Ferry                    483
Aerial Observation       366
Public Aircraft          337
Executive/corporate      316
Skydiving                167
Flight Test              117
Banner Tow                98
Glider Tow                34
Air Race/show             28
Firefighting              18
Air Drop                   5
Name: count, dtype: int64 

Broad.phase.of.flight
NaN            15721
Landing        

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
# Share of missing values per column
missing_share = df.isna().mean().sort_values(ascending=False)
print(missing_share.head(10).round(3))

# Drop columns that are more than 50% empty
cols_to_drop = missing_share[missing_share > 0.5].index
print("\nDropping:", list(cols_to_drop))
df = df.drop(columns=cols_to_drop)
df.shape

Schedule                 0.893
Air.carrier              0.833
FAR.Description          0.692
Longitude                0.611
Latitude                 0.611
Airport.Code             0.387
Airport.Name             0.359
Broad.phase.of.flight    0.286
Publication.Date         0.166
Purpose.of.flight        0.095
dtype: float64

Dropping: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude']


(54990, 30)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
df.to_csv('Aviation_Data_Cleaned.csv', index=False)

# check that it can be loaded back
pd.read_csv('Aviation_Data_Cleaned.csv').shape

(54990, 30)